# Specimen and Geolocation Records of African Bats from Museum Collections (2024) Exploration with `mlcroissant`
This notebook provides a tutorial for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.grgq-s5jj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and sample records.

In [ ]:
# List all record sets and their field IDs
record_sets = dataset.record_sets()
print(f"Record sets in the dataset:\n")
for rs in record_sets:
    print(f"@id: {rs.id_}, name: {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id_}, name: {field.name}")
    print("")

# Show a sample record from the first record set
if record_sets:
    example_record_set = record_sets[0].id_
    print(f"Sample record from record set '{example_record_set}':")
    gen = dataset.records(record_set=example_record_set)
    for i, rec in enumerate(gen):
        print(rec)
        if i == 0:
            break

## 3. Data Extraction
Load records from each record set into a DataFrame for analysis. All entity references by `@id`.

In [ ]:
# Collect all record set @id fields
record_set_ids = [rs.id_ for rs in dataset.record_sets()]
# Prepare a dictionary to hold DataFrames per record set
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Fields in DataFrame for record set '{rs_id}': {df.columns.tolist()}")
    # Show a preview
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter records by a numeric field (for example, 'Year'), normalize values, and group by a categorical field (e.g., 'Country'). All entity references by `@id`.

**Note:** Numeric and grouping fields are chosen below by their `@id` as inferred from the Croissant schema, which you can update based on your overview.

In [ ]:
# Inspect available record set and pick one for EDA
if record_set_ids:
    record_set_id = record_set_ids[0]  # Use the first (main) record set by @id
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}")
    print(f"Available columns: {df.columns.tolist()}")

    # Guess column IDs for numeric and grouping fields (update these if necessary)
    # Ideally, get from field.id_ (e.g., 'Year' might be 'cr:Year')
    possible_numeric = [col for col in df.columns if 'Year' in col or 'year' in col or 'Latitude' in col or 'Longitude' in col or df[col].dtype != 'O']
    numeric_field = possible_numeric[0] if possible_numeric else df.columns[0]
    print(f"Using numeric field '@id': {numeric_field}")

    # Filtering (e.g., records with Year > 2000)
    threshold = 2000 if 'Year' in numeric_field or 'year' in numeric_field else df[numeric_field].mean()
    filtered_df = df[df[numeric_field].apply(pd.to_numeric, errors='coerce') > threshold]  # Convert to numeric if needed
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce') - filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').mean()
    ) / (filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').std())
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping (e.g., by @id containing 'Country' or similar)
    group_fields = [c for c in df.columns if 'Country' in c or 'country' in c or 'Family' in c or 'Genus' in c]
    group_field = group_fields[0] if group_fields else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
if not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If longitude and latitude columns exist, plot a scatter
    lat_candidates = [c for c in df.columns if 'lat' in c.lower()]
    lon_candidates = [c for c in df.columns if 'lon' in c.lower()]
    if lat_candidates and lon_candidates:
        plt.figure(figsize=(6, 6))
        plt.scatter(
            df[lon_candidates[0]].apply(pd.to_numeric, errors='coerce'),
            df[lat_candidates[0]].apply(pd.to_numeric, errors='coerce'),
            alpha=0.6
        )
        plt.title("Specimen Locations (Longitude vs Latitude)")
        plt.xlabel(lon_candidates[0])
        plt.ylabel(lat_candidates[0])
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and analyze the FAIR² African bat specimen dataset. We:

- Loaded dataset metadata and record set schema via Croissant.
- Inspected available record sets, fields, and their entity `@id`s.
- Loaded tabular record data to DataFrames and applied filtering, normalization, and grouping analyses by field `@id`.
- Visualized key numeric fields and (if available) mapped specimen geolocations.

For further study, users may explore more detailed field-based filtering, add domain-specific analyses, or export cleaned data as needed.